# 4k context extension — NeoBERT stage-2 analog + PPPL-vs-length figure

Forks the parked `milestone_5000000_decay` snapshot and continues at **sequence length 4096** over a
fresh, previously-unseen CulturaX window (regrouped from the existing seq-1024 cache — no
re-streaming), at a **constant LR 1e-5** for a planner-derived ~200M-token budget. Then reproduces
NeoBERT **Figure 2**: pseudo-perplexity vs sequence length, before (decay) vs after (ctx4096).

Two hardware stages: **A100** for the 4k finetune, **L4** is enough for the PPPL eval. Re-run safe —
training resumes from the ctx4096 checkpoint. Gate cells fail fast BEFORE any GPU spend.

In [ ]:
%%capture
# Clean, CONSISTENT reinstall — run this FIRST, before anything imports huggingface_hub.
# Colab's base image can carry a half-upgraded huggingface_hub whose new _snapshot_download.py
# imports IncompleteSnapshotError from an older errors.py that lacks it -> `import transformers`
# dies with ImportError. --force-reinstall --no-cache-dir guarantees every hub file comes from ONE
# freshly downloaded wheel (a plain -U reuses cached wheels and leaves the files mixed).
!pip install -U --force-reinstall --no-cache-dir "huggingface_hub>=1.13.0"
!pip install -U --no-cache-dir transformers accelerate datasets safetensors sentencepiece tokenizers pandas matplotlib tqdm
# NO xformers: NeoBERT's fused SwiGLU NaNs on some torch/CUDA builds; artifacts use a patched pure-torch SwiGLU.
!pip uninstall -y xformers

In [ ]:
# Auto-restart ONCE so the kernel loads the freshly reinstalled huggingface_hub. If hub was already
# imported (or its files were swapped under a running kernel), stale modules stay in memory until a
# restart. The flag file lives in /content and survives the restart, so this never loops.
import os
_HF_FLAG = '/content/_hf_reinstalled.flag'
if not os.path.exists(_HF_FLAG):
    open(_HF_FLAG, 'w').close()
    print('huggingface_hub reinstalled -> restarting runtime. When it reconnects, just Run All again.')
    os.kill(os.getpid(), 9)   # Colab auto-restarts the kernel; re-running skips this (flag now exists)
else:
    print('huggingface_hub already reinstalled this session -> continuing without restart.')

In [ ]:
# HF auth (only needed if the eval streams VN-Wikipedia)
from huggingface_hub import login
try:
    from google.colab import userdata
    login(token=userdata.get('HF_TOKEN'))
except Exception:
    import os
    if os.environ.get('HF_TOKEN'):
        login(token=os.environ['HF_TOKEN'])

In [ ]:
import os, sys, importlib
os.environ.setdefault('PYTORCH_CUDA_ALLOC_CONF', 'expandable_segments:True')  # curb fragmentation at seq 4096
from pathlib import Path
import torch

try:
    from google.colab import drive; drive.mount('/content/drive')
except Exception as e:
    print('Drive mount skipped:', e)

PROJECT_ROOT = Path('/content/drive/MyDrive/SALT3') if Path('/content/drive/MyDrive').exists() else Path.cwd() / 'SALT3'
sys.path.insert(0, '/content'); sys.path.insert(0, str(PROJECT_ROOT / 'code'))  # project code FIRST so a stale /content/*.py never shadows the synced modules

import salt3_common as sc; importlib.reload(sc)
import salt3_staged_schedule as sched; importlib.reload(sched)
import salt3_ctx4096 as cx; importlib.reload(cx)
import salt3_pppl_curve as pc; importlib.reload(pc)
from salt3_common import configure_environment, set_seed, ensure_dir

configure_environment(); set_seed(42)
INIT_ROOT     = PROJECT_ROOT / 'init'
RUNS_ROOT     = ensure_dir(PROJECT_ROOT / 'runs' / 'wsd')
DATASET_CACHE = ensure_dir(PROJECT_ROOT / 'datasets')
FIGURES_ROOT  = ensure_dir(PROJECT_ROOT / 'figures')
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
import transformers
print('torch', torch.__version__, '| transformers', transformers.__version__, '| device', DEVICE)
print('salt3_ctx4096  ->', cx.__file__)
print('salt3_pppl_curve->', pc.__file__)

## Config

In [ ]:
INIT_NAME    = 'trung_salt_decpertoken_freezealigned'   # same arm as the decay run (nb19)
CEILING_DOCS = 6_000_000        # seq-1024 cache we regroup the unseen tail from. The 5M cache is
                                # EXHAUSTED (trunk 4M + decay ~0.67M consumed ~all 4.67M train chunks),
                                # so the 4096 stage needs a LARGER cache for fresh tokens -> build it
                                # once below (shared prefix is byte-identical, so contiguity holds).
GROUP        = 4                # 4 x seq-1024 -> seq-4096
TARGET_TOKENS = 200_000_000     # ~200M-token stage-2 budget; planner clamps to the unseen tail
PEAK_LR      = 1e-5             # 10% of the 1e-4 CPT peak; constant plateau after warmup
WARMUP_STEPS = 50
# OOM knob. NeoBERT's SDPA fallback (xformers removed) materializes the full O(seq^2) attention, so
# GPU memory scales as per_device_bs * seq^2. The seq-1024 decay ran bs=32 (=32*1024^2 units); at
# seq 4096 that same footprint is bs=2. bs=4 (=2x decay, ~40GB/80GB) is the speed/safety default;
# drop to 2 if you still OOM, or raise if you have headroom. EFF_BATCH stays 128 either way, so the
# planned window/steps are unchanged — only the accumulation granularity changes.
PER_DEVICE_BS = 4
GRAD_ACCUM    = 32
EVAL_RATIO = 0.02; SEED = 42; MLM_PROB = 0.20
EFF_BATCH  = PER_DEVICE_BS * GRAD_ACCUM

arm_dir     = RUNS_ROOT / INIT_NAME
decay_dir   = arm_dir / 'milestone_5000000_decay'      # the fork point (from nb19)
ctx4096_dir = arm_dir / 'milestone_5000000_ctx4096'    # this run's output
FIG_PATH    = FIGURES_ROOT / 'pppl_vs_length_5m.png'
JSONL_PATH  = FIGURES_ROOT / 'pppl_curve_5m.jsonl'
print('fork point :', decay_dir)
print('output     :', ctx4096_dir)

## Gate — fail fast BEFORE any GPU spend

Four preconditions; each aborts with a fix note if unmet. Guards the decay checkpoint presence, the
RoPE freqs precompute at 4096, the patched-rotary/SwiGLU NaN history at the extended length, and the
window arithmetic (so we know the achieved token budget before committing the A100).

In [ ]:
# Gate 1 — the decay fork point exists on Drive
from transformers import AutoTokenizer
assert (decay_dir / 'model.safetensors').exists(), (
    f'missing {decay_dir}/model.safetensors — run the fresh-data decay (nb19) first and sync to Drive')
tok = AutoTokenizer.from_pretrained(INIT_ROOT / INIT_NAME / 'model', trust_remote_code=True)
print('Gate 1 OK: decay checkpoint present; tokenizer vocab =', tok.vocab_size)

In [ ]:
# Gate 2 — the model config covers seq 4096 (RoPE freqs precompute); else abort with a fix note.
# Take the MAX of both candidate keys: NeoBERT sets max_length=4096, but on configs that omit it,
# HF defaults max_length=20 (generation) which must not mask a valid max_position_embeddings.
from transformers import AutoConfig
cfg = AutoConfig.from_pretrained(decay_dir, trust_remote_code=True)
cands = [getattr(cfg, k, None) for k in ('max_length', 'max_position_embeddings')]
max_len = max([c for c in cands if isinstance(c, int)], default=None)
assert max_len is not None and max_len >= 4096, (
    f'config max length {max_len} < 4096 (candidates {cands}); NeoBERT precomputes rotary freqs to '
    'max_length — raise it in the config and re-save the init before extending context')
print(f'Gate 2 OK: config max length = {max_len} (>= 4096)')

In [ ]:
# Gate 3 — one 4096-token forward returns FINITE logits + loss (patched rotary broadcast + SwiGLU at 4096).
# Mirror NeoMLMTrainer._neo_forward_with_loss: forward WITHOUT labels, read .logits, compute CE ourselves
# (NeoBERT's forward has **kwargs and does not reliably return .loss from labels).
import torch.nn.functional as F
from transformers import AutoModelForMaskedLM
_m = sc.load_model_safe(decay_dir, model_cls=AutoModelForMaskedLM, device=DEVICE).eval()
_ids = torch.randint(low=max(5, (tok.pad_token_id or 0) + 1), high=tok.vocab_size,
                     size=(1, 4096), device=DEVICE)
with torch.no_grad():
    _out = _m(input_ids=_ids, attention_mask=torch.ones_like(_ids))
_logits = _out.logits if hasattr(_out, 'logits') else _out['logits']
assert torch.isfinite(_logits).all(), ('4096-token forward produced non-finite logits — rotary/SwiGLU '
    'patch not applied at the extended length; do NOT train')
_loss = F.cross_entropy(_logits.reshape(-1, _logits.size(-1)), _ids.reshape(-1))
assert torch.isfinite(_loss), f'non-finite 4096-token loss {_loss}; do NOT train'
print(f'Gate 3 OK: 4096-token forward loss = {float(_loss):.4f} (finite)')
del _m, _ids, _out, _logits, _loss
if torch.cuda.is_available(): torch.cuda.empty_cache()

### One-time — build/extend the ceiling cache

The 4096 stage needs FRESH tokens past the decay's consumed position, and the 5M cache is exhausted
(trunk + decay consumed ~all of it). This cell materializes the larger `CEILING_DOCS` cache
(streams CulturaX — **~1hr the first time**, instant reloads after) and prefix-hash-checks that its
shared prefix is byte-identical to the 5M cache, so the decay's chunk index stays contiguous. Skip if
the cache is already built (the call just reloads).

In [ ]:
import hashlib
from salt3_common import make_mlm_datasets, read_cache_meta

# materialize the CEILING_DOCS cache (a chunk_end=1 slice triggers + caches the full build if absent)
make_mlm_datasets(tokenizer=tok, cache_dir=DATASET_CACHE, num_examples=CEILING_DOCS,
                  max_seq_len=cx.BASE_SEQ_LEN, eval_ratio=EVAL_RATIO, seed=SEED, chunk_start=0, chunk_end=1)

def _prefix_hash(num_examples, n=2000):
    tr, _ = make_mlm_datasets(tokenizer=tok, cache_dir=DATASET_CACHE, num_examples=num_examples,
                              max_seq_len=cx.BASE_SEQ_LEN, eval_ratio=EVAL_RATIO, seed=SEED,
                              chunk_start=0, chunk_end=n)
    h = hashlib.sha256()
    for row in tr:
        h.update(repr(list(row['input_ids'])).encode())
    return h.hexdigest()

# contiguity guard: only meaningful while the exhausted 5M cache is still on disk
_meta5 = read_cache_meta(DATASET_CACHE, tok, 5_000_000, cx.BASE_SEQ_LEN, EVAL_RATIO, SEED)
if _meta5 is not None and CEILING_DOCS != 5_000_000:
    assert _prefix_hash(5_000_000) == _prefix_hash(CEILING_DOCS), \
        'cache prefix drift -> decay chunk position NOT contiguous in the new cache; do NOT train'
    print(f'prefix-hash OK: decay chunk position is contiguous in the {CEILING_DOCS:,}-doc cache')
_m = read_cache_meta(DATASET_CACHE, tok, CEILING_DOCS, cx.BASE_SEQ_LEN, EVAL_RATIO, SEED)
print(f'ceiling cache ready: {_m['train_chunks']:,} train 1024-chunks (num_examples={CEILING_DOCS:,})')

In [ ]:
# Gate 4 — read the fork point's consumed position and print the planned window + token budget
import json
from salt3_common import read_cache_meta
decay_plan = decay_dir / 'decay_plan.json'
assert decay_plan.exists(), (
    f'missing {decay_plan}; the ctx4096 window must start past the decay window to stay unseen. '
    'Re-run the decay so it writes decay_plan.json, or set chunks_consumed manually below.')
chunks_consumed = json.loads(decay_plan.read_text())['chunk_end']   # ctx4096 tail starts where decay ended

meta = read_cache_meta(DATASET_CACHE, tok, CEILING_DOCS, cx.BASE_SEQ_LEN, EVAL_RATIO, SEED)
assert meta is not None, f'no cache_meta for ceiling {CEILING_DOCS:,}; build the ceiling cache first'
win = cx.plan_ctx4096_window(chunks_consumed=chunks_consumed, train_chunks_1024=meta['train_chunks'],
                             group=GROUP, eff_batch=EFF_BATCH, target_tokens=TARGET_TOKENS)
print(f'chunks_consumed (decay end) = {chunks_consumed:,}')
print(f'unseen 1024-chunks          = {win["unseen_1024_chunks"]:,}')
print(f'planned steps               = {win["steps"]:,}  (want {win["want_steps"]:,})')
print(f'window (1024-chunks)        = [{win["chunk_start"]:,}, {win["chunk_end"]:,})')
print(f'achieved tokens             = {win["achieved_tokens"]/1e6:.0f}M  ({win["achieved_frac"]*100:.0f}% of target)')
assert win['steps'] >= 1, 'no unseen data for a 4096 run; extend the ceiling cache (raise CEILING_DOCS)'
print('Gate 4 OK: window is single-pass, no-repeat, within the unseen tail')

## Train — 4k context extension (A100)

In [ ]:
res = cx.run_ctx4096_extend(
    arm_dir=arm_dir, base_snapshot=decay_dir, tokenizer=tok, dataset_cache=DATASET_CACHE,
    ceiling_docs=CEILING_DOCS, chunks_consumed=chunks_consumed, group=GROUP,
    eval_ratio=EVAL_RATIO, seed=SEED, per_device_bs=PER_DEVICE_BS, grad_accum=GRAD_ACCUM,
    target_tokens=TARGET_TOKENS, peak_lr=PEAK_LR, warmup_steps=WARMUP_STEPS,
    mlm_probability=MLM_PROB, resume=True, device=DEVICE)
print('ctx4096 checkpoint ->', res['out_dir'])
print('window:', res['window'])
print('final_lr:', res['final_lr'], '| post-eval:', res['post_eval'])

## Figure — pseudo-perplexity vs sequence length (NeoBERT Fig-2 scatter)

One PPPL point per sequence, same (sequence, length) pairs for both checkpoints, drawn as the paper's
two-panel before/after scatter. **Crash-safe & resumable:** each point is written to the JSONL as it's
computed and a re-run skips what's already done — if Colab drops the session, just re-run this cell and
it continues. bf16 + a VRAM-scaled batch keep it fast on any GPU (A100 » L4). To start fresh after
changing `n_docs`/`n_masks`, delete the JSONL first. Raise `n_docs` for a denser cloud (paper ~2,467),
`n_masks` for a crisper per-point PPPL.

In [ ]:
# delete JSONL to start fresh; leave it to resume a crashed run
res_pppl = pc.run_pppl_figure(
    before_dir=decay_dir, after_dir=ctx4096_dir, save_path=FIG_PATH, jsonl_path=JSONL_PATH,
    tokenizer=tok, n_docs=300, min_len=1000, max_len=4096, n_masks=32, seed=SEED, device=DEVICE)
print('figure ->', FIG_PATH)
print('raw points ->', JSONL_PATH)

In [ ]:
# Display the saved figure inline
from IPython.display import Image, display
display(Image(filename=str(FIG_PATH)))

In [ ]:
# Free the GPU when done
try:
    from google.colab import runtime
    runtime.unassign()
except Exception:
    pass